# IBR 动态建模与仿真

九个教学单元。求解器只依赖 Python 标准库；不需要安装 ibrsim。低频实验使用额定频率代数网络与理想电流 / 电压实现，不等同于完整阶次模型。每次运行保留参数、端口、残差及适用范围。网页版本提供交互图。


In [ ]:
"""IBR course teaching models; Python standard library only.

Adapted conventions/control laws from PINN-IBR, reviewed 2026-10-03.
These are independently authored teaching realizations, not ibrsim models.
Network: balanced nominal-frequency algebraic impedances. GFL: ideal current
actuator + SRF PLL (4 states); droop: ideal voltage source + PQ filters (3);
VSM: adds rotor speed (4). Parallel: both branches share one PCC (7).
Switch: explicit GFL -> VSM reset, with voltage and frequency matching.
LCL: separate open-loop averaged six-state electrical plant in seconds.
No switching ripple, DC dynamics, saturation, faults, or protection.
"""
import math
import cmath

WB = 2 * math.pi * 50
NAMES = {"gfl": ["delta", "xi_pll", "id", "iq"],
         "droop": ["delta", "Pf", "Qf"],
         "vsm": ["delta", "omega", "Pf", "Qf"]}


def bounded(value, low, high, name):
    value = float(value)
    if not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f"{name} must be finite and in [{low}, {high}]")
    return value


def settings(case):
    mode = case.get("mode", "droop")
    if mode not in ("frame", "lcl", "gfl", "droop", "vsm", "parallel", "switch", "compare"):
        raise ValueError("Unknown model mode")
    event = case.get("event", "p")
    if event not in ("p", "q", "v", "f"):
        raise ValueError("Unknown disturbance")
    limit = .01 if mode == "lcl" else .5 if event == "f" else .08
    return dict(mode=mode, event=event,
                step=bounded(case.get("step", .005 if mode == "lcl" else .03), -limit, limit, "step"),
                scr=bounded(case.get("scr", 5), 2, 10, "SCR"),
                mp=bounded(case.get("mp", .02), .005, .04, "m_p"),
                inertia=bounded(case.get("inertia", 4), .5, 8, "M"),
                bandwidth=bounded(case.get("bandwidth", 2), .5, 5, "PLL bandwidth"),
                angle=bounded(case.get("angle", 20), -180, 180, "frame angle"),
                dt=bounded(case.get("dt", .0005), .0001, .001, "time step"),
                duration=bounded(case.get("duration", 4), 4, 20, "duration"))


def rk4(rhs, x, h):
    a = rhs(x)
    b = rhs([v + h * d / 2 for v, d in zip(x, a)])
    c = rhs([v + h * d / 2 for v, d in zip(x, b)])
    d = rhs([v + h * e for v, e in zip(x, c)])
    return [v + h * (aa + 2 * bb + 2 * cc + dd) / 6
            for v, aa, bb, cc, dd in zip(x, a, b, c, d)]


def high_voltage_power_flow(s, z):
    """V = 1 + Z conj(S/V), choosing the high-voltage root."""
    a, b = (z * s.conjugate()).real, (z * s.conjugate()).imag
    discriminant = 1 + 4 * a - 4 * b * b
    if discriminant <= 0:
        raise ValueError("No qualified high-voltage equilibrium")
    magnitude_squared = (1 + 2 * a + math.sqrt(discriminant)) / 2
    return complex(magnitude_squared - a, b)


class TeachingCase:
    def __init__(self, kind, cfg):
        self.cfg = cfg
        self.kinds = ["gfl", "droop"] if kind == "parallel" else ["gfl" if kind == "switch" else kind]
        self.zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
        self.zf = complex(.00625, .1)
        self.e0, self.commands, self.slices, self.x0 = [], [], [], []
        voltage = high_voltage_power_flow(complex(.6, 0), self.zg)
        current = complex(.6 / len(self.kinds), 0).conjugate() / voltage.conjugate()
        for family in self.kinds:
            start = len(self.x0)
            if family == "gfl":
                local = current * cmath.exp(-1j * cmath.phase(voltage))
                state = [cmath.phase(voltage), 0, local.real, local.imag]
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), 0.])
            else:
                source = voltage + self.zf * current
                state = [cmath.phase(source), .6 / len(self.kinds), 0]
                if family == "vsm":
                    state.insert(1, 1.)
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), (abs(source) - 1) / .0325])
            self.x0.extend(state)
            self.slices.append((start, len(self.x0)))

    def evaluate(self, x, after=False):
        cfg = self.cfg
        step = cfg["step"] if after else 0
        voltage_grid = 1 + (step if cfg["event"] == "v" else 0)
        omega_grid = 1 + (step / 50 if cfg["event"] == "f" else 0)
        dp = step / len(self.kinds) if cfg["event"] == "p" else 0
        dq = step / len(self.kinds) if cfg["event"] == "q" else 0
        forcing, denominator, entries = complex(voltage_grid), complex(1), []
        for j, (family, (start, stop)) in enumerate(zip(self.kinds, self.slices)):
            xb = x[start:stop]
            if family == "gfl":
                value = complex(xb[2], xb[3]) * cmath.exp(1j * xb[0])
                forcing += self.zg * value
            else:
                qf = xb[3] if family == "vsm" else xb[2]
                amplitude = self.e0[j] - .0325 * (qf - self.commands[j][1] - dq)
                if not .5 < amplitude < 1.5:
                    raise ValueError("Voltage source left the teaching model domain")
                value = amplitude * cmath.exp(1j * xb[0])
                forcing += self.zg * value / self.zf
                denominator += self.zg / self.zf
            entries.append(value)
        voltage = forcing / denominator
        if not .5 < abs(voltage) < 1.5:
            raise ValueError("PCC voltage left the teaching model domain")
        derivatives, currents, powers, omegas = [], [], [], []
        for j, (family, (start, stop), value) in enumerate(zip(self.kinds, self.slices, entries)):
            xb = x[start:stop]
            current = value if family == "gfl" else (value - voltage) / self.zf
            s = voltage * current.conjugate()
            p_ref, q_ref = self.commands[j][0] + dp, self.commands[j][1] + dq
            if family == "gfl":
                local = voltage * cmath.exp(-1j * xb[0])
                error = local.imag / abs(voltage)
                wn = 2 * math.pi * cfg["bandwidth"]
                omega = 1 + 2 * .707 * wn / WB * error + wn * wn / WB * xb[1]
                ref = complex(p_ref, -q_ref) / local.conjugate()
                derivative = [WB * (omega - omega_grid), error,
                              (ref.real - xb[2]) / .02, (ref.imag - xb[3]) / .02]
            elif family == "droop":
                omega = 1 - cfg["mp"] * (xb[1] - p_ref)
                derivative = [WB * (omega - omega_grid), (s.real - xb[1]) / .1, (s.imag - xb[2]) / .05]
            else:
                omega = xb[1]
                derivative = [WB * (omega - omega_grid),
                              (p_ref - xb[2] - (omega - 1) / cfg["mp"]) / cfg["inertia"],
                              (s.real - xb[2]) / .1, (s.imag - xb[3]) / .05]
            derivatives.extend(derivative)
            currents.append(current)
            powers.append(s)
            omegas.append(omega)
        current = sum(currents)
        total = voltage * current.conjugate()
        residual = abs(voltage - voltage_grid - self.zg * current)
        return dict(dx=derivatives, voltage=voltage, currents=currents, powers=powers,
                    p=total.real, q=total.imag, v=abs(voltage), frequency=50 * omegas[0],
                    residual=residual, omega=omegas[0])

    def reset_to_vsm(self, x):
        before = self.evaluate(x, True)
        source = before["voltage"] + self.zf * before["currents"][0]
        s, omega = before["powers"][0], before["omega"]
        dp = self.cfg["step"] if self.cfg["event"] == "p" else 0
        dq = self.cfg["step"] if self.cfg["event"] == "q" else 0
        self.commands[0][0] = s.real + (omega - 1) / self.cfg["mp"] - dp
        self.e0[0] = abs(source) + .0325 * (s.imag - self.commands[0][1] - dq)
        self.kinds[0] = "vsm"
        reset = [cmath.phase(source), omega, s.real, s.imag]
        after = self.evaluate(reset, True)
        error = max(abs(after["voltage"] - before["voltage"]), abs(after["p"] - before["p"]),
                    abs(after["q"] - before["q"]), abs(after["omega"] - omega))
        return reset, error


def controller_run(kind, cfg):
    model = TeachingCase(kind, cfg)
    x = model.x0[:]
    initial_residual = max(map(abs, model.evaluate(x)["dx"]))
    time, traces = [], {key: [] for key in ("p", "q", "v", "frequency")}
    if kind == "parallel":
        traces.update(p_gfl=[], p_gfm=[])
    network_residual, reset_error = 0., None
    # Events land exactly on step boundaries; each segment has frozen inputs.
    segments = [(0., 1., False), (1., 2. if kind == "switch" else cfg["duration"], True)]
    if kind == "switch":
        segments.append((2., cfg["duration"], True))
    for segment, (start, stop, after) in enumerate(segments):
        if kind == "switch" and segment == 2:
            x, reset_error = model.reset_to_vsm(x)
        count = math.ceil((stop - start) / cfg["dt"])
        h = (stop - start) / count
        stride = max(1, round(.01 / h))
        for k in range(count + 1):
            if k % stride == 0 or k == count:
                e = model.evaluate(x, after)
                network_residual = max(network_residual, e["residual"])
                time.append(stop if k == count else start + k * h)
                for key in ("p", "q", "v", "frequency"):
                    traces[key].append(e[key])
                if kind == "parallel":
                    traces["p_gfl"].append(e["powers"][0].real)
                    traces["p_gfm"].append(e["powers"][1].real)
            if k < count:
                x = rk4(lambda state: model.evaluate(state, after)["dx"], x, h)
    manifest = [f"{family}.{name}" for family in model.kinds for name in NAMES[family]]
    return dict(time=time, traces=traces, mode=kind, state_count=len(x), state_names=manifest,
                initial_residual=initial_residual, network_residual=network_residual,
                reset_error=reset_error, final_commands=model.commands,
                scope="nominal-frequency algebraic network; ideal current/voltage realization")


def frame_run(cfg):
    offset = cfg["angle"] * math.pi / 180
    time = [k / 10000 for k in range(401)]
    traces = {key: [] for key in ("va", "vb", "vc", "vd", "vq", "p", "q")}
    residual = 0.
    for t in time:
        phases = [WB * t + shift for shift in (0, -2 * math.pi / 3, 2 * math.pi / 3)]
        va = [math.sqrt(2 / 3) * math.cos(p) for p in phases]
        ia = [math.sqrt(2 / 3) * .6 * math.cos(p - math.pi / 6) for p in phases]
        vd = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, va))
        vq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, va))
        id_ = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, ia))
        iq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, ia))
        p, q = vd * id_ + vq * iq, vq * id_ - vd * iq
        residual = max(residual, abs(p - sum(v * i for v, i in zip(va, ia))))
        for key, value in zip(traces, [*va, vd, vq, p, q]):
            traces[key].append(value)
    return dict(time=time, traces=traces, mode="frame", state_count=0, initial_residual=residual,
                network_residual=0., reset_error=None, scope="balanced power-invariant Park transform")


def lcl_run(cfg):
    r1, ell1, cf = .00625, 8.4375e-5, .0008
    zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
    r2, ell2 = .00625 + zg.real, (.1 + zg.imag) / WB
    i2 = complex(.6)
    vc = 1 + complex(r2, WB * ell2) * i2
    i1 = i2 + 1j * WB * cf * vc
    u = vc + complex(r1, WB * ell1) * i1
    x = [i1.real, i1.imag, vc.real, vc.imag, i2.real, i2.imag]
    def evaluate(state, after):
        a, v, b = complex(*state[:2]), complex(*state[2:4]), complex(*state[4:])
        command = u + (cfg["step"] if after else 0)
        da = (command - v - r1 * a) / ell1 - 1j * WB * a
        dv = (a - b) / cf - 1j * WB * v
        db = (v - 1 - r2 * b) / ell2 - 1j * WB * b
        denergy = ell1 * (a.conjugate() * da).real + cf * (v.conjugate() * dv).real + ell2 * (b.conjugate() * db).real
        balance = (command * a.conjugate()).real - b.real - r1 * abs(a)**2 - r2 * abs(b)**2
        return [da.real, da.imag, dv.real, dv.imag, db.real, db.imag], abs(denergy - balance)
    initial = max(map(abs, evaluate(x, False)[0]))
    time, traces, residual = [], {key: [] for key in ("i2d", "vcd", "vcq")}, 0.
    for start, stop, after in [(0., .01, False), (.01, .03, True)]:
        count = round((stop - start) / .000002)
        h = (stop - start) / count
        for k in range(count + 1):
            if k % 25 == 0 or k == count:
                time.append(stop if k == count else start + k * h)
                for key, value in zip(traces, [x[4], x[2], x[3]]):
                    traces[key].append(value)
                residual = max(residual, evaluate(x, after)[1])
            if k < count:
                x = rk4(lambda state: evaluate(state, after)[0], x, h)
    return dict(time=time, traces=traces, mode="lcl", state_count=6, initial_residual=initial,
                network_residual=residual, reset_error=None, scope="open-loop six-state averaged LCL; energy balance audit")


def solve(case):
    cfg = settings(case)
    if cfg["mode"] == "frame":
        result = frame_run(cfg)
    elif cfg["mode"] == "lcl":
        result = lcl_run(cfg)
    elif cfg["mode"] == "compare":
        runs = {kind: controller_run(kind, cfg) for kind in ("gfl", "droop", "vsm", "parallel")}
        first = runs["gfl"]
        result = dict(time=first["time"], traces={kind: run["traces"]["p"] for kind, run in runs.items()},
                      mode="compare", state_count=0, initial_residual=max(r["initial_residual"] for r in runs.values()),
                      network_residual=max(r["network_residual"] for r in runs.values()), reset_error=None,
                      runs={kind: {k: v for k, v in run.items() if k not in ("time", "traces")} for kind, run in runs.items()},
                      scope=first["scope"])
    else:
        result = controller_run(cfg["mode"], cfg)
    result["case"] = cfg
    return result



## C1-01 · 系统边界与模型表示

### 这个问题，究竟需要多复杂的模型？

一个并网项目拿到了四套逆变器模型，也得到了四条不同的阶跃曲线。团队准备据此选择控制方案。但在比较优劣之前，必须先回答：每条曲线究竟能解释什么？本讲先界定研究问题，再决定模型边界。

### 先明确要解释的现象

<p>同一台变流器可以在开关周期、电气暂态或较慢的同步动态上被研究。若关心开关纹波，就要保留开关动作；若关心 LCL 振荡，就要保留电感和电容的储能；若关心数秒内的小功率指令响应，低阶同步模型可能已足够。</p><p>建模的取舍发生在“需要保留的机理”与“决策需要的证据”之间。因此，先写清观察量、扰动和时间尺度，再选择控制家族。GFM 是控制概念，并不能单独确定模型阶次。</p>





### 先画端口，再数状态

<p>沿平均变流器电压，依次经过 L₁、电容、L₂、并网点 PoC、电网阻抗，最后到达无穷大母线。电容端口与 PoC 之间的支路会储能和耗能，因此，未说明测量端口的功率数值并不完整。</p><p>本课程规定电流流向电网，正 P、Q 表示在声明端口向电网注入。无穷大母线设定电压和频率，直流侧采用理想电源。这些假设共同界定了实验能够支持的并网结论。</p>





### 把边界写成模型契约

<p>把模型分成微分状态 x、代数网络变量 z、输入 u 和输出 y。某个量之所以是状态，是因为它有自己的演化方程，而不是因为它出现在曲线上。网络约束必须在每个时刻与控制器一致。</p><p>三状态 droop 实验采用 x = [δ, P_f, Q_f]。PCC 电压和支路电流是代数量，功率指令和电网条件是输入。电容电压没有被保留。这也直接解释了：该模型无法复现第 3 讲的六状态 LCL 谐振。</p>

$$
\dot{x}=f(x,z,u),\qquad 0=g(x,z,u),\qquad y=h(x,z,u)
$$

### 把案例算清楚

完整 droop 实现有 13 个状态，浏览器中的 droop 实现有 3 个。减少的状态来自哪里？

1. 电气状态有 6 个：i₁、v_c、i₂ 各有 d、q 两个分量。
2. PI 积分状态有 4 个：电压环两个、电流环两个。
3. 留下角度与两个功率滤波状态，构成三状态模型。

13 − 3 = 10 个状态被移除。减少的是电气储能与内环控制的动态及其耦合，并非只把状态向量缩短。

### 同一端口，不同轨迹

四种模型从 P = 0.6 pu、Q = 0 出发，1 s 的 +0.03 pu 指令揭示各自保留的动态。



1. 事件前四条曲线都为 0.6 pu，说明匹配的是端口工作点。
2. 事件后轨迹分开，差异来自保留的同步规律与执行器动态。
3. 这些低阶模型都未保留 LCL 状态，因此不能用它们解释电气谐振。

1. 运行默认比较，指出每条曲线保留了哪些动态。
2. 改为电压阶跃，解释为什么这些轨迹不能检验开关纹波精度。
3. 写出四行模型约定：边界、状态、输入、测量输出。


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


上述 13 状态 droop 模型简化为 3 状态时，移除了多少个状态？

13 − 3 = 10 个状态被移除。减少的是电气储能与内环控制的动态及其耦合，并非只把状态向量缩短。

在三状态 droop 实验中，哪个量是代数量？

1. 滤波有功功率
2. PCC 电压
3. 内部角度

<details><summary>解释</summary>

PCC 电压由网络约束得到，P_f 和 δ 有微分方程。如果显式保留电容，该电容电压就是状态；分类取决于模型表示。

</details>


本讲交付一份四行模型契约：研究问题、测量端口、保留状态、排除机理。用它限定四模型曲线可以支持的结论。

模型边界确定以后，下一个风险来自约定：坐标与基值不同，两条看似正确的方程可能描述不同的量。


## C1-02 · 坐标变换与标幺约定

### 坐标变了，为什么功率不能变？

两位工程师处理同一组三相波形，得到不同的 dq 电流；其中一位用正 i_q 表示正无功注入，另一位用负 i_q。若不先核对变换约定，团队很容易把坐标差异误判为控制问题。

### 改变描述，保留物理功率

<p>在同步旋转坐标中，平衡三相量可以变成近似常量，从而更容易识别电气与控制动态。本课采用功率不变 Park 变换：系数为 √(2/3)，d 行使用 cos，q 行使用 −sin。</p><p>在平衡子空间内，转置矩阵就是逆变换。该变换保留内积，因此 P 的表达式不再乘 3/2；零序则需要额外坐标轴。混用不同来源的约定，即使每条原始公式都正确，组合起来也会出错。</p>





$$
T(\theta)=\sqrt{\frac{2}{3}}\begin{bmatrix}\cos\theta&\cos(\theta-2\pi/3)&\cos(\theta+2\pi/3)\\-\sin\theta&-\sin(\theta-2\pi/3)&-\sin(\theta+2\pi/3)\end{bmatrix}
$$

### 从测量电压反求电流指令

<p>当电流方向定义为向电网注入时，P = v_d i_d + v_q i_q，Q = v_q i_d − v_d i_q。若 d 轴与电压对齐，正 Q 对应负 i_q。这一符号直接来自 q 轴的选择。</p><p>同时求逆两条方程，才能得到电流指令。在 PLL 尚未对齐时，只要电压幅值非零，完整逆映射仍然成立。直接使用 i_d = P/v_d、i_q = −Q/v_d，相当于额外假设整个暂态中 v_q 始终为零。</p>

$$
P=v_d i_d+v_q i_q,\qquad Q=v_q i_d-v_d i_q
$$

$$
\begin{bmatrix}i_d^*\\i_q^*\end{bmatrix}=\frac{1}{v_d^2+v_q^2}\begin{bmatrix}v_d&v_q\\v_q&-v_d\end{bmatrix}\begin{bmatrix}P^*\\Q^*\end{bmatrix}
$$

### 标幺化之后，量纲仍然存在

<p>采用 10 kVA、400 V 线电压 RMS 和 50 Hz，得到 Z_b = 16 Ω。功率不变 dq 电压基值为 400 V，dq 电流基值为 25 A；实际线电流 RMS 基值为 14.434 A。两种电流量不能互换。</p><p>动态系数 ℓ = L/Z_b 与 c = CZ_b 的单位都是秒，额定电抗 x = ω_bℓ 则是另一种量。同样，标幺频率差必须乘 ω_b，才成为角速度差。检查量纲，是识别“看起来很合理”的建模错误的有效方法。</p>

$$
Z_b=\frac{V_b^2}{S_b},\quad I_{b,dq}=\frac{S_b}{V_b},\quad \dot\delta=\omega_b(\omega_{pu}-\omega_{g,pu})
$$

### 把案例算清楚

基频为 50 Hz，相对频率差为 0.001 pu。求相对角度的变化率。

1. 先还原频率差：50 × 0.001 = 0.05 Hz。
2. 每周为 2π 弧度，角速度差为 2π × 0.05。
3. 把这个角速度差代入相对角度微分方程。

角度变化率为 0.314159 rad/s。一秒钟约偏移 18°，足以显著影响同步控制器。

### 分量改变，功率不变

固定 20° 偏移共同旋转电压与电流；不变量是 P、Q，而非不同坐标下的电压分量。



1. 在 +20° 坐标偏差下，电压出现非零 q 分量，但物理电压并未改变。
2. 电压与电流共同旋转，因此功率内积保持不变。
3. 改变实验坐标角度，同时检查 P、Q，不能只看某个坐标分量。

1. 将坐标偏差设为 0°、+20°、−90°，比较 v_d、v_q。
2. 选择 P–Q 曲线，验证物理功率保持不变。
3. 在 frame_run() 中修改电流相角，先预测 P、Q 的符号，再运行。


In [ ]:
case = {"mode": "frame", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


在 50 Hz 基频下，+0.001 pu 频率差对应多大的角度变化率？输入 rad/s。

角度变化率为 0.314159 rad/s。一秒钟约偏移 18°，足以显著影响同步控制器。

v_d = 1、v_q = 0 时，哪个电流注入 Q = +0.2 pu？

1. i_q = +0.2 pu
2. i_q = 0.3 pu
3. i_q = −0.2 pu

<details><summary>解释</summary>

在这个对齐坐标中 Q = −v_d i_q，因此 i_q = −0.2。答案依赖已声明的坐标变换和注入方向；改变任一约定，都需要同步调整整套方程。

</details>


将电压和电流共同旋转 20°，解释为什么 v_d、v_q 改变，而 P、Q 保持不变。每张 dq 曲线都应附上 Park 归一化与电流方向。

坐标与量纲统一后，才能无歧义地推导储能方程。下一讲把物理电路转化为六个电气状态。


## C1-03 · 平均变流器与 LCL 电路

### 电气振荡究竟从哪里来？

一个很小的变流器电压阶跃，引起了快速电流振荡，而较慢的功率控制尚未来得及响应。理想电流源无法解释这一现象。本讲保留 LCL 电路，沿能量流向建立动态。

### 储能元件决定电气状态

<p>每个电感电流与电容电压都需要演化方程。在 dq 坐标下，i₁、v_c、i₂ 各有两个分量，共六个电气状态。网侧滤波电感与电网电感流过相同 i₂，因此可以串联合并后写微分方程。</p><p>这一合并并不意味着 PoC 消失：若要测量端口电压，应通过滤波支路重建。开环实验在 10 ms 给平均变流器 d 轴电压施加 0.005 pu 阶跃，采用 30 ms 窗口和 2 μs 积分步长观察快速响应。</p>





### 正确加入旋转坐标耦合项

<p>先对电感写 KVL、对电容节点写 KCL，再转换到旋转坐标。采用 J = [[0, 1], [−1, 0]] 时，每个向量都增加 +ω_bωJ 乘自身的项。这些跨轴项来自坐标旋转，并不产生或消耗能量。</p><p>此处 ℓ = L/Z_b、c = CZ_b 是时间系数。若把额定电抗误作 ℓ，导数会相差 ω_b 倍。还要区分电气输入电压 u 与控制层功率指令：控制器先生成 u，电路方程再决定实际电流。</p>

$$
\dot i_1=\frac{u-v_c-r_1i_1}{\ell_1}+\omega_b\omega Ji_1
$$

$$
\dot v_c=\frac{i_1-i_2}{c}+\omega_b\omega Jv_c,\qquad\dot i_2=\frac{v_c-v_g-r_2i_2}{\ell_2}+\omega_b\omega Ji_2
$$

$$
v_{PoC}=v_c-r_{f2}i_2-\ell_{f2}(\dot i_2-\omega_b\omega Ji_2)
$$

### 先正确初始化，再判断响应

<p>本电气实验在无穷大母线端口设 i₂ = 0.6 + j0。依次计算 v_c = 1 + (r₂ + jω_bℓ₂)i₂、i₁ = i₂ + jω_bcv_c，以及 u = v_c + (r₁ + jω_bℓ₁)i₁。它们一起满足稳态电路。</p><p>把所有电流设为零，实际上是在仿真上电过程；把 i₁ 设为 i₂，则遗漏了电容电流。这个差别也说明：本讲 LCL 案例在无穷大母线端口初始化，后续低阶模型比较则在 PoC 匹配功率。</p>

### 用能量关系检查推导

<p>定义 W = (ℓ₁|i₁|² + c|v_c|² + ℓ₂|i₂|²)/2，再用电路方程求导。向量与其 J 旋转结果正交，因此旋转项相消，剩下输入功率减去母线输出功率与电阻损耗。</p><p>这条关系能同时核对多个符号。小能量变化率残差说明方程自洽，不能验证已忽略的 PWM 或器件限制。在电气状态上加入两个功率滤波、一个角度、四个 PI 积分状态，就得到源仓库中的 13 状态 droop 实现。</p>

$$
\dot W=u^Ti_1-v_g^Ti_2-r_1\lVert i_1\rVert^2-r_2\lVert i_2\rVert^2
$$

### 把案例算清楚

额定转速下，设 v_c = [1, 0]、c = 0.0008 s，求稳态电容电流导致的 i₁q − i₂q。

1. 令电容方程中的 v̇_c = 0。
2. J[1,0] = [0,−1]，所以 i₁ − i₂ = −ω_bcJv_c。
3. q 分量为 +2π × 50 × 0.0008。

结果为 +0.251327 pu。这个给定电压的算例解释了为什么变流器侧与网侧电流指令不能直接相等。

### 观察电气时间尺度

10 ms 的 0.005 pu d 轴电压阶跃激发开环电路，联合读取 i₂d 与电容电压。



1. 10 ms 变流器电压阶跃是输入，电感电流随后按演化方程响应。
2. 能量在储能元件间交换，电容电压与输出电流共同响应。
3. 用能量变化率关系核对符号，振荡曲线本身无法检查方程。

1. 运行 +0.005 pu 变流器电压阶跃，观察事件后 20 ms。
2. 将 SCR 从 5 降为 2 再运行，说明电气支路中哪个参数改变了。
3. 检查 lcl_run()，识别每项储能，并验证能量变化率残差。


In [ ]:
case = {"mode": "lcl", "scr": 5, "step": 0.005, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


按上述给定 v_c、c，输入 i₁q − i₂q，单位 pu。

结果为 +0.251327 pu。这个给定电压的算例解释了为什么变流器侧与网侧电流指令不能直接相等。

这个开环 LCL 实验包含哪些状态？

1. 两个电流、一个电压，各有 d、q 分量
2. 六个控制积分状态
3. 只有有功与无功功率

<details><summary>解释</summary>

六个状态是 i₁d、i₁q、v_cd、v_cq、i₂d、i₂q。功率由电气变量计算；加入显式测量滤波器后，滤波功率才成为独立状态。

</details>


通过储能与耦合解释振荡。将能量变化率残差与电流曲线一起报告，并指出若要研究闭环阻尼，还必须加入哪些内环。

电路如何响应电压已经明确。下一步要回答：GFL 如何获得参考角度，并生成电流指令？


## C1-04 · PLL 与跟网型控制

### 电网角度改变时，GFL 跟随的是什么？

电网频率上升 0.1 Hz，但功率指令保持不变。要预测响应，需要沿测量电压、PLL 角度、电流注入这条链路逐步推理，而不是直接套用“频率变了，功率就应变化”的直觉。

### 先获取角度，再请求电流

<p>SRF PLL 把端口电压表达在估计坐标中，并将局部 v_q 调到零。正 v_q 表示电压矢量领先估计 d 轴，因此应提高估计角度以减小误差。随后，功率指令映射为局部电流，并旋转到网络坐标。</p><p>四状态教学模型保留相对 PLL 角度 δ、积分 ξ、两个电流执行器分量；20 ms 的理想执行器替代电流 PI 与 LCL。这能单独观察同步与功率跟踪，同时移除了第 3 讲的快速电气动态。</p>





### 把设计频率转成 PLL 参数

<p>把相位检测器归一化为 e = v_q/|V|，并把 PI 输出定义为标幺频率。相对角度方程中再乘 ω_b。在刚性电压、接近对齐时，特征多项式为 s² + ω_bk_p s + ω_bk_i。</p><p>与 s² + 2ζω_n s + ω_n² 对应，得到下面的参数。本实验取 ζ = 0.707、ω_n = 2πb。因此 b 是以 Hz 表示的自然频率设计参数；考虑网络反馈后，它不一定等于实测闭环 −3 dB 带宽。</p>

$$
e=\frac{v_q}{\lVert v\rVert},\quad\dot\xi=e,\quad\hat\omega=1+k_pe+k_i\xi,\quad\dot\delta=\omega_b(\hat\omega-\omega_g)
$$

$$
k_p=\frac{2\zeta\omega_n}{\omega_b},\qquad k_i=\frac{\omega_n^2}{\omega_b}
$$

### 通过电网阻抗闭合反馈

<p>完整 P–Q 逆映射使用两个测量电压分量。网络随后给出 V = V_g + Z_gI，而同一电压又送回 PLL，因此注入电流会影响控制器自己的角度测量。降低 SCR 会加强这种交互。</p><p>对于 +0.1 Hz 扰动，PLL 应跟踪新的频率。本教学控制器没有 frequency–watt 或 RoCoF 功率规律，因此频率跟踪本身不意味着稳态 P 指令发生变化。需要联合观察估计频率、P 与 PCC 电压。</p>

$$
\tau_i\dot i_{dq}=i_{dq}^*-i_{dq},\qquad V=V_g+Z_gI
$$

### 接回物理电流控制器

<p>完整 LCL 实现先将 PoC 功率映射为 i₂*，通过电容电流补偿形成 i₁*，再由电流 PI、电压前馈和电感解耦生成变流器电压 u。实际电流由电气方程决定。</p><p>完整 15 状态 GFL 包含：一个角度、一个 PLL 积分、两个电压测量、一个频率测量、两个功率指令滤波、两个电流 PI 积分和六个电气状态。因此，四状态响应有界，并不能证明完整模型的内环稳定。</p>

$$
i_1^*=i_2^*-\omega_b\hat\omega cJv_c,\quad\dot\xi_i=i_1^*-i_1
$$

$$
u=v_c-\omega_b\hat\omega\ell_1Ji_1+K_{pi}(i_1^*-i_1)+K_{ii}\xi_i
$$

### 把案例算清楚

取 b = 2 Hz、ζ = 0.707、f_b = 50 Hz，计算归一化 PLL 的比例增益。

1. 自然角频率 ω_n = 2π × 2。
2. 基准角频率 ω_b = 2π × 50。
3. 代入 k_p = 2ζω_n/ω_b，2π 相消。

k_p = 0.05656。若相位检测器归一化或 PI 输出单位改变，这个增益也必须相应调整。

### 频率跟踪首先是角度估计任务

本图在 1 s 将电网频率提高 0.1 Hz，PLL 估计随之收敛；下方交互实验初始采用 P 指令基准。



1. 1 s 电网频率提高，角度误差驱动 PLL 修正。
2. 暂时的频率超调属于角度估计反馈的动态。
3. 估计频率收敛至 50.1 Hz；本模型保持 P*，没有额外 frequency–watt 响应。

1. 运行基准有功阶跃，再选择估计频率与 PCC 幅值。
2. 施加 +0.1 Hz 电网频率阶跃，检查暂态后的 PLL 频率。
3. 在 SCR = 2 时，用相同扰动比较 1 Hz 和 5 Hz 的 PLL 设计值。


In [ ]:
case = {"mode": "gfl", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


按本讲归一化 PLL，输入 b = 2 Hz、ζ = 0.707 时的 k_p。

k_p = 0.05656。若相位检测器归一化或 PI 输出单位改变，这个增益也必须相应调整。

局部 v_q 为正时，哪种 PLL 初始动作符合这里的约定？

1. 降低估计角速度
2. 提高估计角速度
3. 自动将 P* 设为零

<details><summary>解释</summary>

k_p 为正时，正 v_q 产生正频率修正，使估计 d 轴向电压向量靠近。这个 PLL 方程本身不会改变功率指令。

</details>


先预测 SCR = 5、+0.1 Hz 扰动的响应，再运行。随后将 SCR 降至 2，通过电压—电流—PLL 反馈解释变化，而不只用 GFL 标签概括。

GFL 从电压获取角度；下一讲的控制器则从功率偏差生成角度，再通过网络实现同步。


## C1-05 · 下垂构网型控制

### 功率偏差如何形成电压角度？

调度将 P* 提高 0.03 pu。GFM 电压源无法直接指定端口功率：它必须先改变频率，再推进角度，由网络传输更多功率。本讲沿着这条闭环解释 droop。

### 功率闭环由网络完成

<p>三状态实现保留 δ、P_f、Q_f。有功滤波偏差设定频率，无功滤波偏差设定电压幅值，再通过积分频率形成内部源角度。内部电压 U = Ee^{jδ} 位于 Z_f = 0.00625 + j0.1 pu 后方。</p><p>正 P* 阶跃发生后，P_f 不能突变，因此频率先升高，角度向前移动，网络传输功率增加，滤波量随后跟上。指令是功率改变的原因，并不直接赋值给实际功率轨迹。</p>





$$
\tau_p\dot P_f=P_{PoC}-P_f,\qquad\tau_q\dot Q_f=Q_{PoC}-Q_f
$$

$$
\omega=1-m_p(P_f-P^*),\quad E=E_0-n_q(Q_f-Q^*),\quad\dot\delta=\omega_b(\omega-\omega_g)
$$

$$
I=\frac{Ee^{j\delta}-V}{Z_f},\qquad P+jQ=VI^*
$$

### 区分端口目标与控制指令

<p>在额定频率锁定的稳态，P_f = P*。无功匹配则更容易被忽视：内部源幅值必须支撑 Z_f 上的电压降。如果 E₀ 固定为 1，即使端口 Q 为零，也需要用 Q* 补偿所需幅值。</p><p>SCR = 5 时，默认匹配得到 Q* 约为 0.316795 pu，而 PoC Q = 0。这是控制器偏置。强行让所有控制器使用相同原始 Q*，会悄悄改变工作点，使比较失去基础。</p>

### 频率扰动揭示稳态斜率

<p>新的锁定稳态中，内部频率等于电网频率。整理 droop 规律得到 P = P* − (ω_g − 1)/m_p。在 50 Hz 下，+0.1 Hz 等于 +0.002 pu；m_p = 0.02 时，预期功率变化为 −0.1 pu。</p><p>同一 m_p 也改变暂态环路增益。当 τ_p = 0.1 s、τ_q = 0.05 s 固定时，峰值与阻尼的变化还反映了网络、滤波和控制的交互。只知道稳态斜率，无法确定暂态形状。</p>

### 问题需要时，再恢复级联内环

<p>完整实现设置 v* = [E,0]，由电压 PI、输出电流前馈和电容解耦生成 i₁*，再由电流 PI 生成 LCL 的输入 u。F 为输出电流前馈增益，K_p、K_i 表示相应 PI 参数。六个电气状态、四个积分、两个功率滤波与一个角度，共 13 个状态。</p><p>源仓库的 canonical 模型用 v_c 与 i₂ 计算滤波功率，而 benchmark 输出在 PoC 重建；浏览器教学实验则特意采用 PoC 反馈。复现完整模型时，应保留原测量端口，并将 PI 积分初始化为支撑稳态电压、电流所需的数值。零跟踪误差并不意味着积分状态为零。</p><p>在未触发限幅、且电压和电流误差为零的稳态，将两条 PI 方程反解为积分状态，得到下式。这些非零偏置用于支撑所需电容电流与变流器电压。若将积分设为零，原本的稳态比较就会混入不必要的启动暂态。</p>

$$
e_v=v^*-v_c,\quad\dot\xi_v=e_v,\quad i_1^*=F i_2-\omega_b\omega cJv_c+K_{pv}e_v+K_{iv}\xi_v
$$

$$
e_i=i_1^*-i_1,\quad\dot\xi_i=e_i,\quad u=v_c-\omega_b\omega\ell_1Ji_1+K_{pi}e_i+K_{ii}\xi_i
$$

$$
\xi_v=\frac{i_1-Fi_2+\omega_b\omega cJv_c}{K_{iv}},\qquad\xi_i=\frac{u-v_c+\omega_b\omega\ell_1Ji_1}{K_{ii}}
$$

### 把案例算清楚

设 P* = 0.6 pu、m_p = 0.02，电网从 50 升至 50.1 Hz。预测锁定后的稳态功率。

1. 标幺频率升幅为 0.1/50 = 0.002。
2. droop 导致的功率降幅为 0.002/0.02 = 0.1 pu。
3. 从指令 0.6 pu 中减去这一降幅。

预测锁定功率为 0.5 pu。这个计算以本模型能够到达新稳态为前提。

### 功率通过角度运动改变

+0.03 pu 的 P* 阶跃先产生频率偏差，再提高传输功率；暂态体现滤波与网络反馈。



1. 指令阶跃时，滤波功率不能突变，因此频率先改变。
2. 源角度前移，通过网络提高功率，测量功率随后反馈。
3. 额定频率稳态要求 P = P*，但需要核对观察窗口是否已足够长。

1. 运行 +0.03 pu P* 阶跃，观察频率并解释为何它先变化。
2. 检查匹配后的 Q*，与初始测量 Q 比较。
3. 施加 +0.1 Hz，比较最终 P 变化与 −0.1/(50 m_p)。


In [ ]:
case = {"mode": "droop", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


对于上述 +0.1 Hz 扰动，输入预测稳态 P，单位 pu。

预测锁定功率为 0.5 pu。这个计算以本模型能够到达新稳态为前提。

端口 Q = 0 时，这个 droop 模型的 Q* 也必须为零吗？

1. 是，对任何网络阻抗都如此
2. 只有 PLL 增益为零时如此
3. 不一定；Q* 可以提供支路所需的内部电压偏置

<details><summary>解释</summary>

Q_f = 0 时，平衡点满足 E = E₀ + n_q Q*。通过 Z_f 的电流要求特定 E，所以 Q* 用于匹配源电压幅值。端口目标与控制器指令是不同的量。

</details>


用一次功率指令阶跃和一次频率阶跃，区分暂态机理与稳态斜率；解释为什么该 droop 同步规律不需要 PLL。

droop 以代数方式设定频率。下一讲增加速度状态，研究虚拟惯量改变了什么，以及还有哪些问题需要额外证据。


## C1-06 · VSM 与虚拟惯量

### 虚拟惯量究竟改变了什么？

设计评审提出“把虚拟惯量加倍，可以改善响应”。要判断这一建议，必须先区分初始变化率、振荡表现、稳态功率，以及设备真实的能量供给能力。

### 给速度增加明确的演化方程

<p>VSM 用 Mω̇ = P* − P_f − D(ω − 1) 替代代数频率 droop，保留 δ、ω、P_f、Q_f。电气电压源与无功规律沿用 droop 实验，因此能单独比较同步规律。</p><p>M 的单位为秒，取 D = 1/m_p。在本摆动方程约定下 M = 2H，默认 M = 4 s 对应 H = 2 s。源仓库区分调速器与显式阻尼贡献，教学模型使用两者合计系数。</p>





$$
M\dot\omega=P^*-P_f-D(\omega-1),\qquad D=\frac1{m_p},\qquad M=2H
$$

### 把初始变化率与稳态分配分开

<p>P* 刚发生阶跃时，速度与滤波功率仍为旧稳态值，因此加速功率为 ΔP*，得到 ḟ = f_bΔP*/M。在同一初始条件下，M 加倍会使初始变化率减半。</p><p>稳态满足 ω̇ = 0、ω = ω_g。当 D = 1/m_p 时，功率—频率斜率与 droop 相同，并不依赖 M。更大惯量会通过与 0.1 s 功率滤波的交互改变振荡与收敛，但不能普遍保证轨迹更好。</p>

$$
\dot\delta=\omega_b(\omega-\omega_g),\qquad P_{\infty}=P^*-\frac{\omega_g-1}{m_p}
$$

$$
\dot f(1^+)=f_b\frac{\Delta P^*}{M}
$$

### 把控制规律转成证据要求

<p>虚拟惯量描述的是控制规律。要真正提供所请求的功率，还需要直流能量来源、可用电流与可实现的变流器电压。实验假设理想直流电源和无限制实现，因此无法证明这些硬件能力。</p><p>基于 PLL 的 GFL 也可以加入滤波 RoCoF 功率支撑，同时保持跟网同步；其结构与 VSM 内部速度状态不同。源仓库 GFL-VI notebook 保留了一个固定参数下不稳定的案例，说明名称中有“惯量”并不能建立稳定性。</p>

### 把案例算清楚

在稳态施加 +0.03 pu 功率指令阶跃，取 M = 4 s、基频 50 Hz，计算初始 RoCoF。

1. 速度导数为 0.03/4 = 0.0075 pu/s。
2. 乘 50 Hz，转换为实际频率变化率。
3. 若 M = 8 s，初始变化率减半。

初始 RoCoF 为 0.375 Hz/s。这一局部预测本身不能确定收敛时间或真实能量供给能力。

### 相同斜率不等于相同暂态

droop 与 VSM 使用相同源、滤波与斜率，本图仅频率规律不同；4 s 窗口未必足以观察收敛。



1. 两种模型收到相同功率指令，源、滤波与稳态斜率也相同。
2. VSM 速度不能突变，新增状态改变了早期轨迹与振荡响应。
3. 4 s 曲线终点只是末次采样，不自动等于稳态值。

1. M = 4 s 时，预测 +0.03 pu 阶跃后的初始频率导数。
2. 固定 m_p、SCR，再分别用 M = 1、8 s 运行。
3. 用频率阶跃实验，对比 droop GFM 的稳态斜率。


In [ ]:
case = {"mode": "vsm", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


输入 ΔP* = 0.03 pu、M = 4 s 时的初始 RoCoF。

初始 RoCoF 为 0.375 Hz/s。这一局部预测本身不能确定收敛时间或真实能量供给能力。

固定 D 和运行点，M 加倍后，同样 P* 阶跃对应的初始响应如何变化？

1. 初始频率导数减半
2. 稳态频率–功率斜率加倍
3. 不再需要直流能量来源

<details><summary>解释</summary>

初始时 Mω̇ = ΔP*，所以 ω̇ 与 M 成反比。平衡时 ω̇ = 0，M 从稳态功率–频率关系中消失。

</details>


在斜率与滤波参数相同的条件下，比较 M = 1、4、8 s，报告初始变化率、峰值，以及是否观察到收敛。判断稳态前将窗口延长至 20 s；若振荡增长，也应保留结果。

现在已有两种同步机制。下一讲把它们同时接入同一个 PCC，控制器之间的交互也随之进入模型。


## C1-07 · GFL–GFM 并联混合模型

### 共享 PCC 为什么不能用两条独立曲线相加？

项目把一个 GFL 支路与一个 droop-GFM 支路并联，两者同时运行。指令增量各分一半看似简单，但共同端口电压会把两条支路的瞬时响应耦合起来。

### 先统一基值，再谈功率分配

<p>两条支路均采用共同的 10 kVA 系统基值，各自从 P = 0.3 pu、Q = 0 出发，PoC 总 P = 0.6 pu。总指令增加 +0.03 pu 时，两条支路各增加 +0.015 pu，仍使用同一基值。</p><p>若实际设备额定容量不同，连接之前应统一功率、电流、阻抗的基值。不同基值下相等的标幺数，并不意味着实际注入相等。</p>





### 求解两个控制器共同看到的网络

<p>GFL 支路提供 I_c，GFM 支路提供 Z_f 后方的 U。由 KCL 得 I_g = I_c + (U − V)/Z_f，再结合 V = V_g + Z_gI_g，得到唯一的 PCC 电压表达式。</p><p>先求出这一电压，再分别变换到各控制器的局部坐标，并计算反馈。七个状态确实由四个 GFL 加三个 droop 组成，但只拼接状态向量并不能连接模型；真正实现连接的是共同网络。</p>

$$
I_g=I_c+\frac{U-V}{Z_f},\qquad V=V_g+Z_gI_g
$$

$$
V=\frac{V_g+Z_g I_c+(Z_g/Z_f)U}{1+Z_g/Z_f}
$$

### 区分分配策略与实际支路功率

<p>在同一个 PCC 计算 S_c = VI_c*、S_v = VI_v*，两者瞬时之和必须等于总复功率。相等的指令增量并不要求暂态输出相等，因为角度动态与电流动态不同。</p><p>降低 SCR 后，联合检查支路与总功率。本讲采用共享 PCC 的电流源／电压源教学实现。源 REGFM_C1 notebook 区分 A11 正序实现与 F21 完整研究扩展；复现其中任一个，都需要对应的方程与参数。</p>

$$
S_{PCC}=V(I_c+I_v)^*=S_c+S_v
$$

### 把案例算清楚

在共同的 10 kVA 基值上，将总 +0.03 pu 功率指令增量平均分给两条支路。

1. 总实际增量为 0.03 × 10 kVA = 300 W。
2. 每条支路分配 150 W。
3. 换回共同基值，150/10000 = 0.015 pu。

每条支路指令增量为共同基值下的 0.015 pu。这并不表示两条支路瞬时响应相等。

### 先有支路交互，再有功率相加

平均分配指令后，共同 PCC 上的支路轨迹仍不同，但每个采样点的和都等于总功率。 图中绘制相对各自初值的功率增量，便于看清小幅支路暂态；下方实验显示实际功率。



1. 两条支路指令都在同一系统基值上提高 0.015 pu。
2. PLL／电流与 droop／角度动态通过共同 V 交互，因此支路轨迹不同。
3. 在每个采样点，两条支路功率之和都等于 PCC 总功率。

1. 选择支路功率，比较 +0.03 pu 总阶跃期间的两个贡献。
2. 检查事件时刻与最终时刻的支路功率之和是否等于总功率。
3. 将 SCR 降至 2 再运行，描述哪个支路响应变化更明显。


In [ ]:
case = {"mode": "parallel", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


输入每条支路在共同基值下的指令增量。

每条支路指令增量为共同基值下的 0.015 pu。这并不表示两条支路瞬时响应相等。

为什么不能直接相加两个独立仿真的支路轨迹？

1. 同一端口的功率不能相加
2. 会遗漏共同 PCC 电压反馈及网络耦合
3. 并联支路必须具有相同状态

<details><summary>解释</summary>

在同一 PCC 计算时，功率可以相加。独立仿真遗漏的是共享电压：每个支路都会改变它，从而改变另一支路的反馈。

</details>


验证每个采样点的支路功率和等于总功率，再选取一段支路响应不同的暂态，通过共同 PCC 反馈和不同同步机制解释。

并联 hybrid 让两种机制同时运行；模式切换 hybrid 则改变当前生效的方程，因此状态含义与切换策略成为关键。


## C1-08 · 模式切换混合模型

### 控制器切换时，究竟什么应该连续？

逆变器在 1 s 扰动后，于 2 s 从 GFL 切换到 VSM。两个简化状态向量恰好都长四维，直接复制似乎很方便，却会把不同物理含义的变量混在一起。

### 模式改变的是生效方程与状态含义

<p>事件之间，当前模式决定 ẋ = f_m(x,u)；切换时，由重置映射决定 x⁺ = R(x⁻,u)。GFL 使用 [δ_PLL, ξ_PLL, i_d, i_q]，VSM 使用 [δ_source, ω, P_f, Q_f]。维数相等，并不意味着状态含义兼容。</p><p>实现中可以保存多个控制器的 checkpoint 并集，但这个并集不会自动变成更高阶 ODE。设计重置时，应从物理端口量出发，而不是从数组位置出发。</p>





$$
\dot x=f_m(x,u),\qquad x^+=R_{m^-\to m^+}(x^-,u)
$$

### 从端口重建进入模式的内部源

<p>切换前记录 V⁻、I⁻、P⁻、Q⁻ 与 PLL 频率。若要使 Z_f 中的端口电流不变，进入模式的内部源必须满足 U⁺ = V⁻ + Z_fI⁻。因此源角度为 arg U⁺，未必等于旧 PLL 角度。</p><p>将 VSM 速度初始化为旧估计速度，功率滤波初始化为测量功率，就能把新模式对齐到旧端口。重置检查应显示电压、电流、功率与内部频率的跳变接近浮点舍入量级。</p>

$$
U^+=V^-+Z_f I^-,\quad\delta^+=\arg U^+,\quad\omega^+=\omega^-,\quad P_f^+=P^-,\quad Q_f^+=Q^-
$$

### 明确切换策略中的取舍

<p>教学策略还选择 E₀⁺ 匹配源幅值，并设置 P*⁺ = P⁻ + D(ω⁻ − 1)，使初始加速度为零。它会重新标定进入模式的指令。若坚持保留旧指令，则属于另一种策略，可能引入加速功率偏差。</p><p>本实验只安排一次切换。可部署设计还需要触发条件、非激活控制器跟踪、驻留规则、限幅与保护。源 transition notebook 进一步研究 hold/release 策略与不同激活状态维数；端点响应良好，本身不能证明切换良好。</p>

$$
E_0^+=|U^+|+n_q(Q^--Q^*),\qquad P^{*+}=P^-+D(\omega^--1)
$$

### 把案例算清楚

在一个给定重置算例中，取 V⁻ = 1 + j0、I⁻ = 0.6 + j0、Z_f = 0.00625 + j0.1 pu，求进入模式的内部源角度。

1. 计算 Z_fI⁻ = 0.00375 + j0.06。
2. 加上端口电压，U⁺ = 1.00375 + j0.06。
3. 用 atan2(0.06, 1.00375)，再从弧度换为角度。

进入模式的源角度约为 3.421°。PLL 可以在端口电压对齐的 0°，而内部源仍然需要非零角度。

### 把扰动与重置分开读

扰动位于 1 s，GFL→VSM 切换位于 2 s。端口兼容的切换应与先前扰动分别解读。



1. 1 s 的指令扰动与 2 s 的模式切换，是两个不同事件。
2. 2 s 时，进入模式通过 U⁺ = V⁻ + Z_fI⁻ 重建原端口。
3. 功率连续还应结合重置检查中的电压、电流与频率共同核对。

1. 运行定时切换，检查 2 s 前后的轨迹。
2. 读取数值检查中的重置残差和最终指令偏置。
3. 在 reset_to_vsm() 中，故意将重置角度改为 PLL 角度，观察跳变。


In [ ]:
case = {"mode": "switch", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


输入上述固定重置算例的内部源角度，单位 °。

进入模式的源角度约为 3.421°。PLL 可以在端口电压对齐的 0°，而内部源仍然需要非零角度。

GFL 与 VSM 状态向量长度相同时，可以直接复制吗？

1. 可以，长度相同就兼容
2. 可以，只要都使用标幺值
3. 不可以；需要按物理含义和端口条件映射

<details><summary>解释</summary>

GFL 的积分量、电流不是 VSM 的速度、滤波功率。应从 V、I 重构接入源，再映射频率和功率测量。重置由状态含义决定，而非向量长度。

</details>


写出连续性检查清单与指令策略。将 2 s 的重置与 1 s 的扰动分别检查，再解释为什么本例不应要求两个角度相等。

最后一讲把这些接口组织成可复现的比较，让曲线成为有依据的工程判断。


## C1-09 · 平衡点、扰动与公平比较

### 什么证据能让模型比较可信？

评审希望知道哪种控制器更好。但四条漂亮曲线还不够：起始工作点、扰动、测量端口和数值精度必须先一致，曲线差异才具有可解释的意义。

### 匹配物理目标，而非所有输入数字

<p>统一采用 10 kVA、400 V、50 Hz 基值、同一个 PoC 和同一个电网。总初始 P = 0.6 pu、Q = 0，并联模型再将目标分到各支路。联立 S = VI* 与 V = 1 + Z_gI，选择潮流高电压解。</p><p>每个控制器都必须在自身坐标中实现这一点。droop、VSM 需要 Q* 偏置来支撑内部电压降。因此，不同原始指令可以对应同一物理目标，这才是我们所需的比较。</p>





$$
S=VI^*,\qquad V=1+Z_g\overline{S/V},\qquad\lVert f(x_0,u_0)\rVert_\infty<\varepsilon
$$

$$
Z_g=\frac{1+j\,10}{\mathrm{SCR}\sqrt{101}},\qquad\Delta P^*_{branch}=\frac{\Delta P^*_{total}}{N}
$$

### 先验证稳态，再准确放置事件

<p>扰动前先检查 max|f(x₀,u₀)|。用旧输入积分到 1 s，再用新输入启动下一段。相同时间戳的前后采样保留突变两侧数值，避免 Runge–Kutta 阶段跨越不连续点进行平均。</p><p>默认步长为 0.5 ms，再用 0.25 ms 重算并在共同输出网格比较轨迹。曲线光滑只是绘图特性；步长细化才提供数值解的证据。切换实验还要在 2 s 增加分段边界与重置。</p>

$$
e_h=\max_k|P_h(t_k)-P_{h/2}(t_k)|
$$

### 区分自洽、有效性与外部验证

<p>稳态、KCL、功率和与重置残差，检查的是与所声明方程是否自洽。与更完整模型在指定范围内比较，评估的是近似有效性。独立测量或经过确认的参考模型，才提供外部验证。</p><p>浏览器实验提供第一类证据。不稳定结果应保留；若 4 s 内无法判断 VSM 收敛，应延长窗口。结论应明确模型、扰动与参数范围，而不是宣布某控制器普遍更优。</p>

### 把模型比较写成决策报告

<p>提交一份简短决策报告，包含模型契约、匹配稳态表、两种受控扰动，以及对响应差异的机理解释；附上参数快照、可执行实验和步长细化结果。</p><p>最后给出与证据相称的建议：哪个实现足以回答当前问题，什么机理解释它的响应，以及扩展结论前还需测试哪些动态。源仓库完整阶次 notebook 提供下一层研究入口。</p>

### 把案例算清楚

在一个稳态匹配算例中，所需源幅值 E = 1.01 pu、E₀ = 1、端口 Q = 0、n_q = 0.0325，求匹配的 Q*。

1. 稳态 Q_f 等于端口 Q，即 0。
2. 整理 E = E₀ − n_q(Q_f − Q*)。
3. 得到 Q* = (1.01 − 1)/0.0325。

匹配指令为 0.307692 pu，而端口 Q 仍然为零。给定源幅值是独立算例，并非默认 SCR = 5 的幅值。

### 先有比较协议，曲线才成为证据

匹配 PoC 功率并施加相同总指令阶跃，才能识别所声明模型的差异；叠图不能建立普遍排名。



1. 所有模型从相同 PoC 总 P、Q 出发，原始控制指令仍可不同。
2. 相同总指令阶跃揭示各模型的暂态机理。
3. 先检查稳态与步长细化，再将差异归因于控制设计。

1. 在 SCR = 5 下，用总 +0.03 pu 有功阶跃比较四个模型。
2. 改为 +0.1 Hz 频率阶跃，解释 GFL 与 GFM 的稳态响应。
3. 在 Python 中将 case["dt"] 减半，对齐采样后比较，并在报告中记录最大误差。


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


输入上述固定稳态算例中的 Q*。

匹配指令为 0.307692 pu，而端口 Q 仍然为零。给定源幅值是独立算例，并非默认 SCR = 5 的幅值。

比较不同控制类别之前，应优先匹配什么？

1. 端口运行点、基值和扰动协议
2. 每一个原始控制指令
3. 只匹配状态数

<details><summary>解释</summary>

不同控制规律可能需要不同内部指令，才能产生同一端口 P、Q。应统一声明的端口条件，并保留各模型自己的状态初始化。原始指令相同不能单独保证公平。

</details>


完成两页决策报告，均衡呈现模型契约、推导、实验与证据强度，并把尚未解决的适用边界写进建议。

建模课程至此闭环。后续 physics-informed 或 neural ODE 代理模型，应继承同样的状态定义、端口约定、稳态与事件协议。
